# RouteRight AI: Incident Reassignment Prediction
## Member 1: Dataset Understanding, Incident-Level Reconstruction, and Target Construction

---

### Project & Module Context
**RouteRight AI** is an intelligent IT Service Management (ITSM) decision-support system designed to predict whether a newly opened IT incident will require **reassignment** to another support group or resolver.

In enterprise ITSM platforms (such as ServiceNow), when an incident is misrouted or bounced across multiple support teams:
1. Mean Time to Resolution (MTTR) increases significantly.
2. Operational costs escalate.
3. SLA compliance drops and user satisfaction decreases.

### Scope of Member 1:
This notebook performs the foundational data engineering and target definition for the project:
1. **Dataset Understanding & Inspection:** Analyzing the structure, attributes, and temporal behavior of the raw IT event log.
2. **Log-to-Incident Reconstruction:** Reconstructing a single, authentic incident-level dataset from raw multi-event log sequences (capturing initial ticket state at creation time).
3. **Target Variable Construction & Initial Leakage Control:** Engineering the ground-truth binary target `reassignment_required` (0 = No Reassignment Required, 1 = Reassignment Required). Direct target leakage from reassignment_count and max_reassignment_count was removed at this stage. Other potentially future-derived fields will be reviewed in the dedicated leakage-validation stage before modelling.

### 1. Import Required Libraries
We import **pandas** for data manipulation and tabular analysis, and **numpy** for numerical operations.


In [1]:
import pandas as pd
import numpy as np

# Set display options for clean and readable tabular outputs
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 1000)


### 2. Load the Raw Incident Event Log Dataset
We load the raw CSV file `incident_event_log.csv` into a pandas DataFrame named `df`.


In [6]:

df = pd.read_csv("../data/raw/incident_event_log.csv")
print("Dataset successfully loaded.")


Dataset successfully loaded.


### 3. Dataset Shape, First Rows, Column Names, Data Types, and Info
We inspect the structural properties of the dataset, including dimensionality, column names, data types, and non-null counts.


In [7]:
# Dataset Shape
print(f"Dataset Shape: {df.shape[0]:,} rows (event records), {df.shape[1]} columns")


Dataset Shape: 141,712 rows (event records), 36 columns


In [4]:
# First 5 rows
df.head()


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 21,29/2/2016 01:23,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 642,29/2/2016 08:53,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 804,29/2/2016 11:29,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 908,5/3/2016 12:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,Created by 171,29/2/2016 04:57,Updated by 746,29/2/2016 04:57,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 89,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


In [5]:
# Column Names
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")


Column Names:
 1. number
 2. incident_state
 3. active
 4. reassignment_count
 5. reopen_count
 6. sys_mod_count
 7. made_sla
 8. caller_id
 9. opened_by
10. opened_at
11. sys_created_by
12. sys_created_at
13. sys_updated_by
14. sys_updated_at
15. contact_type
16. location
17. category
18. subcategory
19. u_symptom
20. cmdb_ci
21. impact
22. urgency
23. priority
24. assignment_group
25. assigned_to
26. knowledge
27. u_priority_confirmation
28. notify
29. problem_id
30. rfc
31. vendor
32. caused_by
33. closed_code
34. resolved_by
35. resolved_at
36. closed_at


In [6]:
# Data Types
print("Data Types:")
print(df.dtypes)


Data Types:
number                       str
incident_state               str
active                      bool
reassignment_count         int64
reopen_count               int64
sys_mod_count              int64
made_sla                    bool
caller_id                    str
opened_by                    str
opened_at                    str
sys_created_by               str
sys_created_at               str
sys_updated_by               str
sys_updated_at               str
contact_type                 str
location                     str
category                     str
subcategory                  str
u_symptom                    str
cmdb_ci                      str
impact                       str
urgency                      str
priority                     str
assignment_group             str
assigned_to                  str
knowledge                   bool
u_priority_confirmation     bool
notify                       str
problem_id                   str
rfc                          st

In [7]:
# DataFrame Summary Info
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 141712 entries, 0 to 141711
Data columns (total 36 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   number                   141712 non-null  str  
 1   incident_state           141712 non-null  str  
 2   active                   141712 non-null  bool 
 3   reassignment_count       141712 non-null  int64
 4   reopen_count             141712 non-null  int64
 5   sys_mod_count            141712 non-null  int64
 6   made_sla                 141712 non-null  bool 
 7   caller_id                141712 non-null  str  
 8   opened_by                141712 non-null  str  
 9   opened_at                141712 non-null  str  
 10  sys_created_by           141712 non-null  str  
 11  sys_created_at           141712 non-null  str  
 12  sys_updated_by           141712 non-null  str  
 13  sys_updated_at           141712 non-null  str  
 14  contact_type             141712 non-null  str  

### 4. Raw Event Records vs. Unique Incidents
The raw dataset is an **event log**, meaning each row corresponds to a logged event/update rather than an individual incident. We verify the total number of event records versus the number of distinct incidents using the `number` identifier.


In [8]:
raw_events_count = len(df)
unique_incidents_count = df['number'].nunique()

print(f"Total Raw Event Records (Rows) : {raw_events_count:,}")
print(f"Total Unique Incidents (Tickets): {unique_incidents_count:,}")


Total Raw Event Records (Rows) : 141,712
Total Unique Incidents (Tickets): 24,918


### 5. Demonstrating Multiple Rows per Incident
To illustrate the event log nature of the data, we display all logged updates for a sample incident (`INC0000045`). Notice how the incident transitions through states (`New`, `Resolved`, `Closed`) and system update timestamps.


In [9]:
sample_incident_id = 'INC0000045'
sample_events = df[df['number'] == sample_incident_id][
    ['number', 'incident_state', 'active', 'reassignment_count', 'sys_updated_at', 'assignment_group', 'assigned_to']
]
sample_events


,number,incident_state,active,reassignment_count,sys_updated_at,assignment_group,assigned_to
0,INC0000045,New,True,0,29/2/2016 01:23,Group 56,?
1,INC0000045,Resolved,True,0,29/2/2016 08:53,Group 56,?
2,INC0000045,Resolved,True,0,29/2/2016 11:29,Group 56,?
3,INC0000045,Closed,False,0,5/3/2016 12:00,Group 56,?


### 6. Calculate and Summarize Event Records per Incident
We calculate how many event log rows exist per unique incident and analyze the distribution (minimum, maximum, mean, quartiles).


In [10]:
events_per_incident = df['number'].value_counts()

print("Summary Statistics: Number of Events per Incident")
print(events_per_incident.describe())


Summary Statistics: Number of Events per Incident
count    24918.000000
mean         5.687134
std          3.677845
min          2.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         58.000000
Name: count, dtype: float64


### 7. Check for Exact Duplicate Rows
We verify whether there are any completely identical duplicate rows across all 36 columns in the raw dataset.


In [11]:
exact_duplicate_count = df.duplicated().sum()
print(f"Number of Exact Duplicate Rows across all columns: {exact_duplicate_count}")


Number of Exact Duplicate Rows across all columns: 0


### 8. Inspect `reassignment_count` Before Target Construction
We examine the distribution and summary statistics of `reassignment_count` in the raw event log. In the raw log, `reassignment_count` increases dynamically as tickets get transferred between groups.


In [12]:
print("Summary Statistics for 'reassignment_count' in Raw Log:")
print(df['reassignment_count'].describe())

print("\nFrequency Distribution of 'reassignment_count' in Raw Log (Top 10):")
print(df['reassignment_count'].value_counts().head(10))


Summary Statistics for 'reassignment_count' in Raw Log:
count    141712.000000
mean          1.104197
std           1.734673
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max          27.000000
Name: reassignment_count, dtype: float64

Frequency Distribution of 'reassignment_count' in Raw Log (Top 10):
reassignment_count
0    69876
1    37104
2    15097
3     8274
4     4614
5     2595
6     1447
7      985
8      574
9      365
Name: count, dtype: int64


### 9. Convert Timestamp Columns to Datetime
We parse `opened_at` (ticket opening time) and `sys_updated_at` (system event update time) to datetime using day-first parsing (`dayfirst=True`) and robust error handling (`errors='coerce'`).


In [13]:
# Convert to datetime using day-first parsing
df['sys_updated_at_dt'] = pd.to_datetime(df['sys_updated_at'], dayfirst=True, errors='coerce')
df['opened_at_dt'] = pd.to_datetime(df['opened_at'], dayfirst=True, errors='coerce')

print("Datetime conversion completed successfully.")
print(f"Missing/Invalid 'sys_updated_at' values: {df['sys_updated_at_dt'].isna().sum()}")
print(f"Missing/Invalid 'opened_at' values: {df['opened_at_dt'].isna().sum()}")


Datetime conversion completed successfully.
Missing/Invalid 'sys_updated_at' values: 0
Missing/Invalid 'opened_at' values: 0


### 10. Chronological Sorting by Incident Number and Update Timestamp
To accurately identify the initial state of each incident, we sort the entire DataFrame chronologically by `number` and the parsed `sys_updated_at_dt`.


In [14]:
df_sorted = df.sort_values(by=['number', 'sys_updated_at_dt'], ascending=[True, True])
print("DataFrame sorted chronologically by incident number and system update timestamp.")


DataFrame sorted chronologically by incident number and system update timestamp.


### 11. Extract Earliest Chronological Row for Each Incident
For each incident, the earliest chronological event row was retained to represent the initial incident state.

> **Methodological Note (Viva defense point):**
> We use `df_sorted.drop_duplicates(subset='number', keep='first')` rather than `df.groupby('number').first()`. 
> `groupby().first()` selects the first *non-null* value per column across all chronological rows, which could accidentally pull updated values from later lifecycle stages. `drop_duplicates(keep='first')` strictly guarantees that the earliest chronological event row was retained to represent the initial incident state without pulling updated values from later stages.

In [15]:
# Extract earliest chronological event row for each incident to represent initial incident state
initial_incidents = df_sorted.drop_duplicates(subset='number', keep='first').copy()
print(f"Initial Incident Records Extracted: {len(initial_incidents):,}")

Initial Incident Records Extracted: 24,918


### 12. Verify Exactly One Row per Unique Incident
We verify that the reconstructed initial incidents dataset contains exactly one record for each unique incident in the raw dataset.


In [16]:
assert len(initial_incidents) == df['number'].nunique(), "Row count does not match unique incidents!"
print(f"Verification Passed: Exactly {len(initial_incidents):,} records extracted for {df['number'].nunique():,} unique incidents.")


Verification Passed: Exactly 24,918 records extracted for 24,918 unique incidents.


### 13. Calculate Maximum `reassignment_count` Across Full Incident History
To determine whether an incident was ever reassigned during its entire lifecycle, we calculate the maximum `reassignment_count` for each incident across all its historical event log records.


In [17]:
# Calculate max reassignment_count per incident across entire log history
max_reassignment = df.groupby('number')['reassignment_count'].max().reset_index()
max_reassignment.rename(columns={'reassignment_count': 'max_reassignment_count'}, inplace=True)

print("Sample of Maximum Historical Reassignment Counts:")
max_reassignment.head(10)


Sample of Maximum Historical Reassignment Counts:


,number,max_reassignment_count
0,INC0000045,0
1,INC0000047,1
2,INC0000057,0
3,INC0000060,0
4,INC0000062,1
5,INC0000063,1
6,INC0000064,1
7,INC0000065,6
8,INC0000066,1
9,INC0000067,1


### 14. Merge Maximum Reassignment Count with Initial Incident Record
We merge the calculated `max_reassignment_count` onto the initial incident DataFrame using `number` as the key.


In [18]:
incident_df = pd.merge(initial_incidents, max_reassignment, on='number', how='left')
print(f"Merged Dataset Shape: {incident_df.shape[0]:,} rows, {incident_df.shape[1]} columns")


Merged Dataset Shape: 24,918 rows, 39 columns


### 15. Create Binary Target Column: `reassignment_required`
We create the binary ground-truth target variable `reassignment_required`:
- **`0`**: No Reassignment Required — No reassignment was recorded during the incident lifecycle (`max_reassignment_count == 0`).
- **`1`**: Reassignment Required — At least one reassignment was recorded during the incident lifecycle (`max_reassignment_count > 0`).

In [19]:
incident_df['reassignment_required'] = (incident_df['max_reassignment_count'] > 0).astype(int)
print("Binary target 'reassignment_required' successfully created.")


Binary target 'reassignment_required' successfully created.


### 16. Verify Target Class Distribution
We inspect the target counts and percentage proportions as a verification step to understand class balance.


In [20]:
target_counts = incident_df['reassignment_required'].value_counts()
target_proportions = incident_df['reassignment_required'].value_counts(normalize=True) * 100

target_distribution = pd.DataFrame({
    'Class': ['0 (No Reassignment Required)', '1 (Reassignment Required)'],
    'Count': [target_counts[0], target_counts[1]],
    'Percentage (%)': [round(target_proportions[0], 2), round(target_proportions[1], 2)]
})
target_distribution.set_index('Class', inplace=True)
print("Target Variable Distribution:")
target_distribution


Target Variable Distribution:


,Count,Percentage (%)
Class,,
0 (No Reassignment Required),13549,54.37
1 (Reassignment Required),11369,45.63


### 17, 18 & 19. Prevent Direct Target Leakage and Clean Temporary Helper Columns
Direct target leakage from reassignment_count and max_reassignment_count was removed at this stage. Other potentially future-derived fields will be reviewed in the dedicated leakage-validation stage before modelling.

Key cleanup steps:
1. **Remove `reassignment_count` and `max_reassignment_count`:** Dropped from predictor data to eliminate direct target leakage.
2. **Remove temporary datetime sorting helpers (`sys_updated_at_dt`, `opened_at_dt`):** Cleans up intermediate calculation columns.
3. **Preserve original `opened_at` field:** Kept intact and preserved for later feature engineering (e.g., extracting hour of day, day of week).

In [21]:
# Define helper and leakage columns to drop
columns_to_drop = ['reassignment_count', 'max_reassignment_count', 'sys_updated_at_dt', 'opened_at_dt']

# Create final incident-level dataset
final_incident_df = incident_df.drop(columns=[col for col in columns_to_drop if col in incident_df.columns]).copy()

print(f"Final Dataset Shape: {final_incident_df.shape[0]:,} rows, {final_incident_df.shape[1]} columns")
print(f"\nFinal Column List ({len(final_incident_df.columns)} columns):")
print(final_incident_df.columns.tolist())


Final Dataset Shape: 24,918 rows, 36 columns

Final Column List (36 columns):
['number', 'incident_state', 'active', 'reopen_count', 'sys_mod_count', 'made_sla', 'caller_id', 'opened_by', 'opened_at', 'sys_created_by', 'sys_created_at', 'sys_updated_by', 'sys_updated_at', 'contact_type', 'location', 'category', 'subcategory', 'u_symptom', 'cmdb_ci', 'impact', 'urgency', 'priority', 'assignment_group', 'assigned_to', 'knowledge', 'u_priority_confirmation', 'notify', 'problem_id', 'rfc', 'vendor', 'caused_by', 'closed_code', 'resolved_by', 'resolved_at', 'closed_at', 'reassignment_required']


### 20. Comprehensive Final Validation
We execute a comprehensive validation checklist to verify data integrity across all key reconstruction criteria.


In [22]:
validation_checks = {
    'Raw Event Records Count': len(df),
    'Raw Unique Incidents Count': df['number'].nunique(),
    'Final Incident-Level Row Count': len(final_incident_df),
    'Final Unique Incident Count': final_incident_df['number'].nunique(),
    'Duplicate Incident Numbers in Final': final_incident_df['number'].duplicated().sum(),
    'Missing Target Values': final_incident_df['reassignment_required'].isna().sum()
}

validation_df = pd.DataFrame([
    {
        'Validation Metric': metric,
        'Value': f"{val:,}" if isinstance(val, int) else str(val),
        'Status': 'PASSED' if (
            (metric == 'Final Incident-Level Row Count' and val == df['number'].nunique()) or
            (metric == 'Final Unique Incident Count' and val == df['number'].nunique()) or
            (metric == 'Duplicate Incident Numbers in Final' and val == 0) or
            (metric == 'Missing Target Values' and val == 0) or
            ('Raw' in metric)
        ) else 'FAILED'
    }
    for metric, val in validation_checks.items()
])

print("Final Reconstruction & Validation Report:")
validation_df


Final Reconstruction & Validation Report:


,Validation Metric,Value,Status
0,Raw Event Records Count,"141,712",PASSED
1,Raw Unique Incidents Count,"24,918",PASSED
2,Final Incident-Level Row Count,"24,918",PASSED
3,Final Unique Incident Count,"24,918",PASSED
4,Duplicate Incident Numbers in Final,0,PASSED
5,Missing Target Values,0,PASSED


### 21. Save Reconstructed Incident-Level Dataset
We save the clean, reconstructed incident-level dataset to `incident_level_dataset.csv`.


In [ ]:
final_incident_df.to_csv("../data/interim/incident_level_dataset.csv", index=False)
print("Successfully exported reconstructed dataset to 'incident_level_dataset.csv'.")


Successfully exported reconstructed dataset to 'incident_level_dataset.csv'.


## Target Construction Summary

### Summary of Member 1 Findings and Implementation

| Item / Concept | Description & Implementation Details |
| :--- | :--- |
| **Raw Dataset Nature** | The raw dataset (`incident_event_log.csv`) is an **event log** with **141,712 rows** containing sequential audit trails and state transitions for IT service incidents. |
| **Multi-Row Relationship** | A single incident generates multiple rows (ranging from **2 to 58 events per incident**, mean = 5.69) as its state, resolver, or attributes are updated over time. |
| **Incident-Level Reconstruction** | The dataset was successfully reconstructed from 141,712 log rows into **24,918 distinct incident records** (exactly one row per unique incident ticket). |
| **Initial Ticket State Extraction** | Records were sorted chronologically by `number` and `sys_updated_at`. For each incident, the earliest chronological event row was retained to represent the initial incident state using `drop_duplicates(subset='number', keep='first')` without pulling updated values from later stages. |
| **Historical Target Derivation** | The maximum `reassignment_count` across each ticket's full lifecycle history was computed separately and merged to establish ground truth. |
| **Target Variable Definition** | A binary target column **`reassignment_required`** was constructed:<br>&bull; **0 = No Reassignment Required** &mdash; No reassignment was recorded during the incident lifecycle (13,549 incidents / **54.37%**).<br>&bull; **1 = Reassignment Required** &mdash; At least one reassignment was recorded during the incident lifecycle (11,369 incidents / **45.63%**). |
| **Direct Target Leakage Removal** | Direct target leakage from `reassignment_count` and `max_reassignment_count` was removed at this stage. Other potentially future-derived fields will be reviewed in the dedicated leakage-validation stage before modelling. |
| **Feature Preservation** | The original `opened_at` timestamp has been preserved for later feature engineering (hour of day, day of week, rush hours) in downstream tasks. |
| **Export File** | The finalized dataset is saved as **`incident_level_dataset.csv`** (24,918 rows &times; 36 columns). |

---
**End of Member 1 Responsibilities.** The reconstructed dataset is saved in `incident_level_dataset.csv` and ready for subsequent workflow phases.